# Group 1: Economic Growth and Quality of Life in East Africa
## World Bank Indicators API - complete analysis notebook

This notebook implements the capstone workflow for 2010-2024. It:

1. requests data directly from the World Bank Indicators API;
2. saves the complete API responses as raw JSON;
3. converts the observations into tidy and wide CSV files;
4. assesses missing values, duplicates, outliers, completeness, and consistency;
5. creates a fully populated analysis-ready dataset using transparent, flagged imputation;
6. reviews every statistical outlier and corrects only values that fail a plausibility rule;
7. performs exploratory analysis plus correlation, regression, and trend analysis;
8. gives a direct, evidence-based answer to each of the six research questions;
9. couples Questions 1-5 with five main visualisations and places the extra trend chart afterwards; and
10. records the endpoints, parameters, variables, statistical results, and chart files used.

The final cleaned CSV has no blank analytical values. Missing API observations are estimated using within-country interpolation or a regional year pattern adjusted to the country. Each estimated cell is explicitly flagged and its method is recorded. Outlier flags are treated as review prompts, not automatic deletion rules.

> **Presentation guide:** Every numbered step ends with one short sentence that explains what was done in plain English. The answer cells also show exactly what to report.


## Step 1 - Save results in Google Drive

Run this cell and approve the Google Drive permission prompt. The notebook will create `MyDrive/EAC_Group1_Capstone/data`.

> **Presentation explanation:** We saved every output in Google Drive so the work is not lost when Colab closes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 - Import libraries

Google Colab normally includes these libraries, so no API key or package installation is required.

> **Presentation explanation:** We imported the tools needed to download, organise, analyse, and display the data.

In [ ]:
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Step 3 - Define the study scope and indicators

The default scope contains the eight current EAC Partner States. If your lecturer gives the group a narrower country list, change only the `COUNTRIES` dictionary before running the retrieval cells. Do not silently remove a country merely because it has missing observations.

Four series are retrieved:

- `NY.GDP.PCAP.KD`: GDP per capita in constant 2015 US dollars. Use this to calculate total real growth from 2010 to 2024.
- `NY.GDP.PCAP.KD.ZG`: annual GDP-per-capita growth percentage. Use this for yearly growth trends.
- `SP.DYN.LE00.IN`: life expectancy at birth, total, in years.
- `SE.SEC.ENRR`: gross secondary-school enrolment percentage. A gross ratio can exceed 100 because it can include over-age and under-age students.

> **Presentation explanation:** We selected the eight EAC countries, the years 2010-2024, and four World Bank indicators.

In [ ]:
START_YEAR = 2010
END_YEAR = 2024
SOURCE_ID = 2  # World Development Indicators
BASE_URL = 'https://api.worldbank.org/v2'

COUNTRIES = {
    'BDI': 'Burundi',
    'COD': 'Democratic Republic of the Congo',
    'KEN': 'Kenya',
    'RWA': 'Rwanda',
    'SOM': 'Somalia',
    'SSD': 'South Sudan',
    'TZA': 'Tanzania',
    'UGA': 'Uganda',
}

INDICATORS = {
    'NY.GDP.PCAP.KD': {
        'column': 'gdp_per_capita_constant_2015_usd',
        'label': 'GDP per capita (constant 2015 US$)',
    },
    'NY.GDP.PCAP.KD.ZG': {
        'column': 'gdp_per_capita_growth_annual_pct',
        'label': 'GDP per capita growth (annual %)',
    },
    'SP.DYN.LE00.IN': {
        'column': 'life_expectancy_years',
        'label': 'Life expectancy at birth, total (years)',
    },
    'SE.SEC.ENRR': {
        'column': 'secondary_enrolment_gross_pct',
        'label': 'School enrollment, secondary (% gross)',
    },
}

COUNTRY_CODES = ';'.join(COUNTRIES.keys())
OUTPUT_DIR = Path('/content/drive/MyDrive/EAC_Group1_Capstone/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Countries: {len(COUNTRIES)}')
print(f'Indicators: {len(INDICATORS)}')
print(f'Years: {START_YEAR}-{END_YEAR}')
print(f'Output folder: {OUTPUT_DIR}')

## Step 4 - Display and save the API documentation table

This table documents the endpoint pattern, parameters, and variables required by the project brief.

> **Presentation explanation:** We recorded the exact API address and settings so another person can repeat the download.

In [ ]:
endpoint_pattern = f'{BASE_URL}/country/{COUNTRY_CODES}/indicator/{{indicator_code}}'
api_parameters = {
    'date': f'{START_YEAR}:{END_YEAR}',
    'format': 'json',
    'source': SOURCE_ID,
    'per_page': 20000,
}

indicator_dictionary = pd.DataFrame([
    {
        'indicator_code': code,
        'indicator_name': info['label'],
        'output_column': info['column'],
        'endpoint': endpoint_pattern.format(indicator_code=code),
        'date_parameter': api_parameters['date'],
        'source_id': SOURCE_ID,
    }
    for code, info in INDICATORS.items()
])

indicator_dictionary.to_csv(OUTPUT_DIR / 'world_bank_group1_indicator_dictionary.csv', index=False)
display(indicator_dictionary)
print('Parameters:', api_parameters)

## Step 5 - Define reliable API request functions

The functions below check HTTP errors, retry temporary failures, validate the JSON structure, and follow pagination.

> **Presentation explanation:** We made the download function retry temporary errors and check that the response is valid.

In [ ]:
session = requests.Session()
session.headers.update({
    'User-Agent': 'Strathmore-MIT8334-EAC-Group1-Capstone/1.0'
})

def request_json(url, params, max_attempts=4, timeout=60):
    """Request JSON and retry temporary connection/server failures."""
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = session.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json(), response.url
        except (requests.RequestException, ValueError) as error:
            last_error = error
            if attempt == max_attempts:
                break
            wait_seconds = 2 ** (attempt - 1)
            print(f'Attempt {attempt} failed; retrying in {wait_seconds}s: {error}')
            time.sleep(wait_seconds)
    raise RuntimeError(f'API request failed after {max_attempts} attempts: {last_error}')

def fetch_indicator(indicator_code):
    """Fetch every page for one indicator and retain the unmodified responses."""
    url = f'{BASE_URL}/country/{COUNTRY_CODES}/indicator/{indicator_code}'
    page = 1
    raw_pages = []
    all_records = []

    while True:
        params = {
            'date': f'{START_YEAR}:{END_YEAR}',
            'format': 'json',
            'source': SOURCE_ID,
            'per_page': 20000,
            'page': page,
        }
        payload, request_url = request_json(url, params)

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(
                f'Unexpected response for {indicator_code}: {str(payload)[:300]}'
            )

        metadata = payload[0] or {}
        records = payload[1] or []
        raw_pages.append({
            'request_url': request_url,
            'metadata': metadata,
            'records': records,
        })
        all_records.extend(records)

        total_pages = int(metadata.get('pages', 1))
        if page >= total_pages:
            break
        page += 1

    return raw_pages, all_records

## Step 6 - Retrieve the World Bank data and save raw JSON

Run this cell while connected to the internet. The World Bank Indicators API does not require an API key.

> **Presentation explanation:** We downloaded every page of data and kept the original JSON as evidence of what the API returned.

In [ ]:
raw_bundle = {
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
    'api_name': 'World Bank Indicators API v2',
    'base_url': BASE_URL,
    'source_id': SOURCE_ID,
    'study_period': {'start_year': START_YEAR, 'end_year': END_YEAR},
    'countries': COUNTRIES,
    'indicators': INDICATORS,
    'responses': {},
}
records_by_indicator = {}
request_log_rows = []

for code, info in INDICATORS.items():
    print(f'Fetching {code}: {info["label"]}')
    raw_pages, records = fetch_indicator(code)
    raw_bundle['responses'][code] = {'pages': raw_pages}
    records_by_indicator[code] = records

    for page_number, page_data in enumerate(raw_pages, start=1):
        request_log_rows.append({
            'indicator_code': code,
            'page': page_number,
            'records_on_page': len(page_data['records']),
            'request_url': page_data['request_url'],
            'last_updated': page_data['metadata'].get('lastupdated'),
        })
    print(f'  received {len(records)} country-year records')

raw_json_path = OUTPUT_DIR / 'world_bank_group1_raw_2010_2024.json'
with raw_json_path.open('w', encoding='utf-8') as file:
    json.dump(raw_bundle, file, ensure_ascii=False, indent=2)

request_log = pd.DataFrame(request_log_rows)
request_log.to_csv(OUTPUT_DIR / 'world_bank_group1_request_log.csv', index=False)

print(f'Raw JSON saved to: {raw_json_path}')
display(request_log)

## Step 7 - Convert the JSON records into a tidy table

A tidy table has one row per country-year-indicator observation. Values are converted to numeric types, but null observations remain null.

> **Presentation explanation:** We changed the API response into a simple table with one observation per row.

In [ ]:
rows = []

for indicator_code, records in records_by_indicator.items():
    for item in records:
        rows.append({
            'country_code': item.get('countryiso3code'),
            'country': COUNTRIES.get(
                item.get('countryiso3code'),
                (item.get('country') or {}).get('value')
            ),
            'year': item.get('date'),
            'indicator_code': indicator_code,
            'indicator_name': (item.get('indicator') or {}).get('value'),
            'variable': INDICATORS[indicator_code]['column'],
            'value': item.get('value'),
            'unit': item.get('unit'),
            'observation_status': item.get('obs_status'),
            'decimal': item.get('decimal'),
        })

long_df = pd.DataFrame(rows)
long_df['year'] = pd.to_numeric(long_df['year'], errors='coerce').astype('Int64')
long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')

long_df = long_df[
    long_df['country_code'].isin(COUNTRIES)
    & long_df['year'].between(START_YEAR, END_YEAR)
].copy()
long_df = long_df.sort_values(
    ['country_code', 'year', 'indicator_code']
).reset_index(drop=True)

print('Shape:', long_df.shape)
display(long_df.head(12))

## Step 8 - Assess duplicates and data completeness

The expected maximum is 15 values per country-indicator combination. A lower number is not automatically an error: World Bank series, especially education data, may not be reported every year.

> **Presentation explanation:** We checked repeated records and counted how many expected values were available.

In [ ]:
key_columns = ['country_code', 'year', 'indicator_code']
duplicate_mask = long_df.duplicated(key_columns, keep=False)
duplicate_rows = long_df.loc[duplicate_mask].copy()

print(f'Duplicate rows on {key_columns}: {duplicate_mask.sum()}')
if not duplicate_rows.empty:
    display(duplicate_rows)

coverage_base = pd.MultiIndex.from_product(
    [COUNTRIES.keys(), INDICATORS.keys()],
    names=['country_code', 'indicator_code']
).to_frame(index=False)

non_missing = long_df.dropna(subset=['value'])
coverage_observed = (
    non_missing.groupby(['country_code', 'indicator_code'])
    .agg(
        non_missing_years=('year', 'nunique'),
        first_available_year=('year', 'min'),
        last_available_year=('year', 'max'),
    )
    .reset_index()
)

coverage = coverage_base.merge(
    coverage_observed,
    on=['country_code', 'indicator_code'],
    how='left',
)
coverage['country'] = coverage['country_code'].map(COUNTRIES)
coverage['indicator_name'] = coverage['indicator_code'].map(
    {code: info['label'] for code, info in INDICATORS.items()}
)
coverage['expected_years'] = END_YEAR - START_YEAR + 1
coverage['non_missing_years'] = coverage['non_missing_years'].fillna(0).astype(int)
coverage['missing_years'] = coverage['expected_years'] - coverage['non_missing_years']
coverage['coverage_pct'] = (
    100 * coverage['non_missing_years'] / coverage['expected_years']
).round(1)
coverage = coverage[[
    'country_code', 'country', 'indicator_code', 'indicator_name',
    'expected_years', 'non_missing_years', 'missing_years',
    'coverage_pct', 'first_available_year', 'last_available_year'
]].sort_values(['indicator_code', 'country_code']).reset_index(drop=True)

display(coverage)

## Step 9 - Create a complete country-year wide table

This creates all 120 country-year combinations (8 countries x 15 years), then adds one column per indicator. A blank cell means the API returned no value for that country-year-series.

> **Presentation explanation:** We created every country-year combination so missing values are visible instead of disappearing.

In [ ]:
# Remove exact key duplicates only after they have been counted above.
long_clean = long_df.drop_duplicates(key_columns, keep='first').copy()

wide_values = (
    long_clean.pivot_table(
        index=['country_code', 'year'],
        columns='variable',
        values='value',
        aggfunc='first',
    )
    .reset_index()
)
wide_values.columns.name = None

country_year_grid = pd.MultiIndex.from_product(
    [COUNTRIES.keys(), range(START_YEAR, END_YEAR + 1)],
    names=['country_code', 'year']
).to_frame(index=False)
country_year_grid['country'] = country_year_grid['country_code'].map(COUNTRIES)

wide_df = country_year_grid.merge(
    wide_values, on=['country_code', 'year'], how='left'
)

indicator_columns = [info['column'] for info in INDICATORS.values()]
for column in indicator_columns:
    if column not in wide_df.columns:
        wide_df[column] = np.nan

wide_df = wide_df[
    ['country_code', 'country', 'year'] + indicator_columns
].sort_values(['country_code', 'year']).reset_index(drop=True)

print('Wide-table shape:', wide_df.shape)
assert len(wide_df) == len(COUNTRIES) * (END_YEAR - START_YEAR + 1)
display(wide_df.head(20))

## Step 10 - Save the CSV datasets and verification summary

> **Presentation explanation:** We saved the tidy, wide, coverage, and verification files for later analysis.

In [ ]:
long_csv_path = OUTPUT_DIR / 'world_bank_group1_tidy_long_2010_2024.csv'
wide_csv_path = OUTPUT_DIR / 'world_bank_group1_clean_ready_wide_2010_2024.csv'
coverage_csv_path = OUTPUT_DIR / 'world_bank_group1_coverage_summary.csv'
duplicates_csv_path = OUTPUT_DIR / 'world_bank_group1_duplicate_rows.csv'

long_clean.to_csv(long_csv_path, index=False)
wide_df.to_csv(wide_csv_path, index=False)
coverage.to_csv(coverage_csv_path, index=False)
duplicate_rows.to_csv(duplicates_csv_path, index=False)

verification = pd.DataFrame({
    'check': [
        'configured countries',
        'study years',
        'configured indicators',
        'expected wide rows',
        'actual wide rows',
        'duplicate API keys before deduplication',
        'missing values in wide indicator columns',
    ],
    'result': [
        len(COUNTRIES),
        END_YEAR - START_YEAR + 1,
        len(INDICATORS),
        len(COUNTRIES) * (END_YEAR - START_YEAR + 1),
        len(wide_df),
        int(duplicate_mask.sum()),
        int(wide_df[indicator_columns].isna().sum().sum()),
    ],
})
verification.to_csv(OUTPUT_DIR / 'world_bank_group1_retrieval_verification.csv', index=False)

display(verification)
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)

## Step 11 - Preview the retrieved data

This quick check confirms that the retrieval stage produced one row per configured country and year before the detailed assessment begins.

> **Presentation explanation:** We previewed one country to confirm that the rows and years were created correctly.

In [ ]:
display(wide_df[wide_df['country_code'] == 'KEN'])

# Part B - Data quality assessment and preprocessing

## Step 12 - Prepare analysis and visualisation libraries

The following sections assess all five required quality dimensions: missing values, duplicate records, outliers, completeness, and consistency.

> **Presentation explanation:** We loaded the charting and statistical packages used in the rest of the notebook.

In [ ]:
import importlib.util
import subprocess
import sys

required_analysis_packages = ['matplotlib', 'seaborn', 'scipy', 'statsmodels']
missing_packages = [
    package for package in required_analysis_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    print('Installing missing analysis packages:', missing_packages)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', *missing_packages
    ])

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import statsmodels.formula.api as smf

# Official Strathmore University brand colours from the brand guide.
SU_BLUE = '#3848A2'
SU_RED = '#E52D2C'
SU_GOLD = '#C79633'
SU_YELLOW = '#F4CE67'
SU_BLACK = '#201D1C'
SU_LIGHT_BLUE = '#EEF0FA'
SU_LIGHT_RED = '#FBE5E4'

sns.set_theme(style='whitegrid', context='notebook', rc={
    'axes.edgecolor': '#D7D9E5',
    'grid.color': '#E6E7EE',
    'text.color': SU_BLACK,
    'axes.labelcolor': SU_BLACK,
    'axes.titlecolor': SU_BLUE,
})
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

country_order = list(COUNTRIES.keys())
strathmore_palette = [
    SU_BLUE, SU_RED, SU_GOLD, '#6F78BE',
    '#F07A74', '#DAB86B', SU_BLACK, '#8E93B8'
]
country_palette = dict(zip(country_order, strathmore_palette))
su_sequential = LinearSegmentedColormap.from_list(
    'strathmore_sequential', ['#FFFFFF', '#D8DCF2', '#858CCB', SU_BLUE]
)
su_diverging = LinearSegmentedColormap.from_list(
    'strathmore_diverging', [SU_RED, '#F8C6C4', '#FFFFFF', '#D8DCF2', SU_BLUE]
)

GDP_LEVEL = 'gdp_per_capita_constant_2015_usd'
GDP_GROWTH = 'gdp_per_capita_growth_annual_pct'
LIFE = 'life_expectancy_years'
EDUCATION = 'secondary_enrolment_gross_pct'
ANALYSIS_COLUMNS = [GDP_LEVEL, GDP_GROWTH, LIFE, EDUCATION]

print('Figures will be saved to:', FIGURE_DIR)

def add_chart_heading(fig, title, subtitle=None, top=0.84):
    """Add a title and subtitle without covering the chart."""
    fig.suptitle(title, y=0.985, fontsize=14, fontweight='bold', color=SU_BLUE)
    if subtitle:
        readable_subtitle = subtitle.replace('; ', ';\n')
        fig.text(0.5, 0.925, readable_subtitle, ha='center', va='top',
                 fontsize=9, color='0.35')
    fig.tight_layout(rect=[0, 0, 1, top])


## Step 13 - Assess missing values and data completeness

Completeness is calculated as the percentage of the 15 study years containing a reported value. Missing means *not reported by the API*; it does not mean zero.

> **Presentation explanation:** We counted blank values and calculated the percentage of years available for each indicator.

In [ ]:
analysis_raw = wide_df.copy()
analysis_raw['year'] = pd.to_numeric(analysis_raw['year'], errors='coerce').astype('Int64')
for column in ANALYSIS_COLUMNS:
    analysis_raw[column] = pd.to_numeric(analysis_raw[column], errors='coerce')

missing_summary = pd.DataFrame({
    'variable': ANALYSIS_COLUMNS,
    'missing_count': [analysis_raw[c].isna().sum() for c in ANALYSIS_COLUMNS],
    'non_missing_count': [analysis_raw[c].notna().sum() for c in ANALYSIS_COLUMNS],
    'missing_pct': [100 * analysis_raw[c].isna().mean() for c in ANALYSIS_COLUMNS],
})
missing_summary['missing_pct'] = missing_summary['missing_pct'].round(1)

completeness_matrix = (
    analysis_raw.groupby(['country_code', 'country'])[ANALYSIS_COLUMNS]
    .agg(lambda series: 100 * series.notna().mean())
    .round(1)
    .reset_index()
)

display(missing_summary)
display(completeness_matrix)

### Data-quality visual - completeness heatmap

A heatmap makes country-indicator coverage gaps immediately visible. A single sequential scale is used because all cells share the same 0-100% unit. This chart supports the quality assessment and is not counted among the six research-question charts below.

In [ ]:
completeness_plot = completeness_matrix.set_index('country')[ANALYSIS_COLUMNS].rename(columns={
    GDP_LEVEL: 'GDP per capita',
    GDP_GROWTH: 'GDP growth',
    LIFE: 'Life expectancy',
    EDUCATION: 'Secondary enrolment',
})

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    completeness_plot, annot=True, fmt='.0f', cmap=su_sequential,
    vmin=0, vmax=100, linewidths=0.5,
    cbar_kws={'label': 'Available study years (%)'}, ax=ax
)
ax.set_title('World Bank data completeness before preprocessing', weight='bold', color=SU_BLUE)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig.savefig(FIGURE_DIR / '00_data_completeness_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Step 14 - Assess duplicate records, outliers, and consistency

Duplicate checks use the business key `country_code + year`. Outliers are flagged within each country's time series using the 1.5 x IQR rule. A flag is a prompt for review, not proof that the value is wrong. We correct a value only when it also fails a broad plausibility rule. Gross secondary enrolment is allowed to exceed 100 because the indicator is a gross ratio.

Consistency checks test identifiers, year range, numeric finiteness, and plausible ranges. Any impossible value is converted to missing, recorded, and then estimated by the same transparent method used for other missing cells.

> **Presentation explanation:** We flagged unusual values, checked whether they were actually impossible, and kept plausible shocks instead of deleting real events.


In [ ]:
wide_key = ['country_code', 'year']
wide_duplicate_mask = analysis_raw.duplicated(wide_key, keep=False)
wide_duplicate_rows = analysis_raw.loc[wide_duplicate_mask].copy()
raw_duplicate_count = int(duplicate_mask.sum()) if 'duplicate_mask' in globals() else 0

outlier_flag_columns = []
for column in ANALYSIS_COLUMNS:
    flag_column = f'{column}_iqr_outlier'
    outlier_flag_columns.append(flag_column)
    analysis_raw[flag_column] = False
    for country_code, index_values in analysis_raw.groupby('country_code').groups.items():
        values = analysis_raw.loc[index_values, column].dropna()
        if len(values) < 4:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        analysis_raw.loc[index_values, flag_column] = (
            (analysis_raw.loc[index_values, column] < lower)
            | (analysis_raw.loc[index_values, column] > upper)
        )

outlier_records = []
for column, flag_column in zip(ANALYSIS_COLUMNS, outlier_flag_columns):
    flagged = analysis_raw.loc[
        analysis_raw[flag_column], ['country_code', 'country', 'year', column]
    ].copy()
    flagged = flagged.rename(columns={column: 'value'})
    flagged['variable'] = column
    outlier_records.append(flagged)
outlier_detail = pd.concat(outlier_records, ignore_index=True)

expected_names = analysis_raw['country_code'].map(COUNTRIES)
consistency_checks = pd.DataFrame([
    {'check': 'Unexpected country code', 'issue_count': int((~analysis_raw['country_code'].isin(COUNTRIES)).sum())},
    {'check': 'Country name does not match configured code', 'issue_count': int(((analysis_raw['country'] != expected_names) & expected_names.notna()).sum())},
    {'check': 'Year outside configured period', 'issue_count': int((~analysis_raw['year'].between(START_YEAR, END_YEAR)).sum())},
    {'check': 'GDP per capita is non-positive', 'issue_count': int((analysis_raw[GDP_LEVEL].notna() & (analysis_raw[GDP_LEVEL] <= 0)).sum())},
    {'check': 'Life expectancy outside 20-100 years', 'issue_count': int((analysis_raw[LIFE].notna() & ~analysis_raw[LIFE].between(20, 100)).sum())},
    {'check': 'Gross secondary enrolment outside 0-200%', 'issue_count': int((analysis_raw[EDUCATION].notna() & ~analysis_raw[EDUCATION].between(0, 200)).sum())},
    {'check': 'Non-finite numeric values', 'issue_count': int((~np.isfinite(analysis_raw[ANALYSIS_COLUMNS].fillna(0))).sum().sum())},
])

quality_summary = pd.DataFrame([
    {'quality_dimension': 'Missing values before cleaning', 'issue_count': int(analysis_raw[ANALYSIS_COLUMNS].isna().sum().sum()), 'decision': 'Estimate transparently; never replace with zero'},
    {'quality_dimension': 'Duplicate records', 'issue_count': raw_duplicate_count + int(wide_duplicate_mask.sum()), 'decision': 'Assess raw and wide business keys; keep first only after review'},
    {'quality_dimension': 'IQR outlier candidates', 'issue_count': int(analysis_raw[outlier_flag_columns].sum().sum()), 'decision': 'Review with plausibility rules; do not delete automatically'},
    {'quality_dimension': 'Consistency issues', 'issue_count': int(consistency_checks['issue_count'].sum()), 'decision': 'Correct impossible values to missing, log, then estimate'},
])

display(quality_summary)
display(consistency_checks)
display(outlier_detail.head(25))


## Step 15 - Create a complete dataset and document every decision

Cleaning and completion decisions:

1. Standardise years and indicators as numeric, sort the panel, and remove confirmed duplicate keys.
2. Convert impossible values to missing and record them before estimating anything.
3. Fill gaps *between* two reported country values with straight-line interpolation.
4. Fill leading or trailing GDP gaps from the regional median for that year, scaled to the country's observed level.
5. Fill leading or trailing life-expectancy and education gaps from the regional median plus the country's typical offset.
6. If a country has no reported values for an indicator, use the regional median for each year.
7. Fill missing GDP-growth rates from the completed GDP-per-capita series.
8. Keep plausible IQR outliers and label the review action; correct only values that fail a plausibility rule.
9. Add method and imputation flags so every estimated cell can be identified.

> **Presentation explanation:** We filled gaps using the country's own trend where possible, used the regional pattern only when needed, and marked every estimated value.


In [ ]:
clean_df = analysis_raw.drop_duplicates(wide_key, keep='first').copy()
clean_df = clean_df.sort_values(['country_code', 'year']).reset_index(drop=True)
clean_df['country'] = clean_df['country_code'].map(COUNTRIES)

# Record the API missingness before any correction or estimation.
original_missing_mask = clean_df[ANALYSIS_COLUMNS].isna().copy()

invalid_conditions = {
    GDP_LEVEL: clean_df[GDP_LEVEL].notna() & (clean_df[GDP_LEVEL] <= 0),
    GDP_GROWTH: clean_df[GDP_GROWTH].notna() & ~clean_df[GDP_GROWTH].between(-100, 100),
    LIFE: clean_df[LIFE].notna() & ~clean_df[LIFE].between(20, 100),
    EDUCATION: clean_df[EDUCATION].notna() & ~clean_df[EDUCATION].between(0, 200),
}
invalid_records = []
for column, condition in invalid_conditions.items():
    bad_rows = clean_df.loc[condition, ['country_code', 'country', 'year', column]].copy()
    bad_rows = bad_rows.rename(columns={column: 'original_value'})
    bad_rows['variable'] = column
    bad_rows['action'] = 'Corrected to missing because the value failed a plausibility rule; then imputed'
    invalid_records.append(bad_rows)
    clean_df.loc[condition, column] = np.nan
invalid_detail = pd.concat(invalid_records, ignore_index=True)

def complete_panel_series(data, column, adjustment, lower=None, upper=None):
    """Fill a country-year series and return values plus a method label for every row."""
    completed = pd.Series(index=data.index, dtype=float)
    methods = pd.Series(index=data.index, dtype='object')
    regional_by_year = data.groupby('year')[column].median()
    global_median = float(data[column].median())

    for country_code, group in data.groupby('country_code', sort=False):
        group = group.sort_values('year')
        series = group.set_index('year')[column].astype(float)
        observed = series.dropna()
        filled = series.interpolate(method='linear', limit_area='inside')
        internal_years = set(series.index[series.isna() & filled.notna()])

        shared_years = [
            year for year in observed.index
            if pd.notna(regional_by_year.get(year, np.nan))
        ]
        if adjustment == 'ratio' and shared_years:
            ratios = [
                observed.loc[year] / regional_by_year.loc[year]
                for year in shared_years if regional_by_year.loc[year] != 0
            ]
            country_adjustment = float(np.median(ratios)) if ratios else 1.0
        elif adjustment == 'difference' and shared_years:
            country_adjustment = float(np.median([
                observed.loc[year] - regional_by_year.loc[year]
                for year in shared_years
            ]))
        else:
            country_adjustment = np.nan

        for year in filled.index[filled.isna()]:
            regional_value = regional_by_year.get(year, np.nan)
            if pd.isna(regional_value):
                regional_value = global_median
            if observed.empty:
                estimate = regional_value
                method = 'regional_year_median_no_country_reports'
            elif adjustment == 'ratio':
                estimate = regional_value * country_adjustment
                method = 'regional_year_median_scaled_to_country'
            else:
                estimate = regional_value + country_adjustment
                method = 'regional_year_median_plus_country_offset'
            filled.loc[year] = estimate
            methods.loc[group.index[group['year'].eq(year)]] = method

        if lower is not None or upper is not None:
            filled = filled.clip(lower=lower, upper=upper)

        for row_index, year in zip(group.index, group['year']):
            completed.loc[row_index] = filled.loc[year]
            if pd.notna(series.loc[year]):
                methods.loc[row_index] = 'reported_world_bank_value'
            elif year in internal_years:
                methods.loc[row_index] = 'within_country_linear_interpolation'

    return completed, methods

clean_df[GDP_LEVEL], clean_df[f'{GDP_LEVEL}_method'] = complete_panel_series(
    clean_df, GDP_LEVEL, adjustment='ratio', lower=1
)
clean_df[LIFE], clean_df[f'{LIFE}_method'] = complete_panel_series(
    clean_df, LIFE, adjustment='difference', lower=20, upper=100
)
clean_df[EDUCATION], clean_df[f'{EDUCATION}_method'] = complete_panel_series(
    clean_df, EDUCATION, adjustment='difference', lower=0, upper=200
)

# Fill only missing GDP-growth values from the completed GDP-per-capita series.
calculated_growth = clean_df.groupby('country_code')[GDP_LEVEL].pct_change(fill_method=None) * 100
growth_missing = clean_df[GDP_GROWTH].isna()
clean_df[f'{GDP_GROWTH}_method'] = 'reported_world_bank_value'
clean_df.loc[growth_missing, GDP_GROWTH] = calculated_growth.loc[growth_missing]
clean_df.loc[growth_missing, f'{GDP_GROWTH}_method'] = 'calculated_from_completed_gdp_per_capita'
remaining_growth_missing = clean_df[GDP_GROWTH].isna()
if remaining_growth_missing.any():
    yearly_growth_median = clean_df.groupby('year')[GDP_GROWTH].transform('median')
    clean_df.loc[remaining_growth_missing, GDP_GROWTH] = yearly_growth_median.loc[remaining_growth_missing]
    clean_df.loc[remaining_growth_missing, f'{GDP_GROWTH}_method'] = 'regional_year_median'

# Imputation flags refer to values missing from the API or invalid values corrected above.
for column in ANALYSIS_COLUMNS:
    invalid_mask = invalid_conditions[column].reindex(clean_df.index, fill_value=False)
    clean_df[f'{column}_was_imputed'] = (
        original_missing_mask[column].reindex(clean_df.index, fill_value=False)
        | invalid_mask
    )
    candidate_column = f'{column}_iqr_outlier'
    clean_df[f'{column}_outlier_candidate'] = clean_df[candidate_column].fillna(False).astype(bool)
    clean_df[f'{column}_outlier_action'] = np.select(
        [
            invalid_mask,
            clean_df[f'{column}_outlier_candidate'],
        ],
        [
            'corrected_invalid_value_then_imputed',
            'reviewed_and_retained_plausible',
        ],
        default='not_flagged',
    )

clean_df['log_gdp_per_capita'] = np.log(clean_df[GDP_LEVEL])
clean_df['complete_core_case'] = clean_df[[GDP_LEVEL, LIFE, EDUCATION]].notna().all(axis=1)

# Build cell-level audit tables outside the clean CSV so the clean CSV itself has no blanks.
imputation_rows = []
for column in ANALYSIS_COLUMNS:
    mask = clean_df[f'{column}_was_imputed']
    for row in clean_df.loc[mask, [
        'country_code', 'country', 'year', column, f'{column}_method'
    ]].itertuples(index=False, name=None):
        imputation_rows.append({
            'country_code': row[0], 'country': row[1], 'year': int(row[2]),
            'variable': column, 'final_value': row[3], 'method': row[4],
            'reason': 'Value was missing or invalid in the API response',
        })
imputation_audit = pd.DataFrame(imputation_rows)

outlier_review_rows = []
for column in ANALYSIS_COLUMNS:
    mask = clean_df[f'{column}_outlier_candidate']
    for row in clean_df.loc[mask, [
        'country_code', 'country', 'year', column, f'{column}_outlier_action'
    ]].itertuples(index=False, name=None):
        outlier_review_rows.append({
            'country_code': row[0], 'country': row[1], 'year': int(row[2]),
            'variable': column, 'reviewed_value': row[3], 'action': row[4],
            'reason': 'Value passed the broad plausibility rule; an IQR flag alone does not prove an error',
        })
outlier_review = pd.DataFrame(outlier_review_rows)

preprocessing_log = pd.DataFrame([
    {'step': 1, 'decision': 'Numeric type standardisation and sorting', 'reason': 'Required for arithmetic and time order', 'effect': 'No observations removed'},
    {'step': 2, 'decision': 'Duplicate key removal', 'reason': 'One row per country-year is required', 'effect': f'{raw_duplicate_count + int(wide_duplicate_mask.sum())} duplicate rows reviewed'},
    {'step': 3, 'decision': 'Plausibility correction', 'reason': 'Impossible values should not enter calculations', 'effect': f'{len(invalid_detail)} values corrected to missing before imputation'},
    {'step': 4, 'decision': 'Within-country interpolation', 'reason': 'The two surrounding country values are the strongest guide for internal gaps', 'effect': 'Internal missing years filled linearly'},
    {'step': 5, 'decision': 'Region-adjusted edge completion', 'reason': 'No later or earlier country observation exists for an endpoint gap', 'effect': 'Regional year pattern scaled or offset to the country'},
    {'step': 6, 'decision': 'No-report country fallback', 'reason': 'A country trend cannot be estimated without any reported value', 'effect': 'Regional year median used and clearly flagged'},
    {'step': 7, 'decision': 'Outlier review', 'reason': 'IQR flags may be real shocks or trend endpoints', 'effect': f'{int(clean_df[[f"{c}_outlier_candidate" for c in ANALYSIS_COLUMNS]].sum().sum())} candidates reviewed; {len(invalid_detail)} confirmed invalid values corrected'},
    {'step': 8, 'decision': 'Log transform GDP for regression', 'reason': 'Reduce right skew', 'effect': 'Adds log_gdp_per_capita'},
])

clean_output_columns = ['country_code', 'country', 'year']
for column in ANALYSIS_COLUMNS:
    clean_output_columns.extend([
        column, f'{column}_was_imputed', f'{column}_method',
        f'{column}_outlier_candidate', f'{column}_outlier_action'
    ])
clean_output_columns.extend(['log_gdp_per_capita', 'complete_core_case'])
clean_df = clean_df[clean_output_columns].copy()

complete_csv_path = OUTPUT_DIR / 'world_bank_group1_cleaned_complete_2010_2024.csv'
clean_df.to_csv(complete_csv_path, index=False)
imputation_audit.to_csv(OUTPUT_DIR / 'world_bank_group1_imputation_audit.csv', index=False)
outlier_review.to_csv(OUTPUT_DIR / 'world_bank_group1_outlier_review.csv', index=False)
missing_summary.to_csv(OUTPUT_DIR / 'world_bank_group1_missing_value_summary.csv', index=False)
quality_summary.to_csv(OUTPUT_DIR / 'world_bank_group1_data_quality_summary.csv', index=False)
completeness_matrix.to_csv(OUTPUT_DIR / 'world_bank_group1_completeness_matrix.csv', index=False)
wide_duplicate_rows.to_csv(OUTPUT_DIR / 'world_bank_group1_wide_duplicate_rows.csv', index=False)
outlier_detail.to_csv(OUTPUT_DIR / 'world_bank_group1_outlier_flags.csv', index=False)
consistency_checks.to_csv(OUTPUT_DIR / 'world_bank_group1_consistency_checks.csv', index=False)
invalid_detail.to_csv(OUTPUT_DIR / 'world_bank_group1_invalid_values_log.csv', index=False)
preprocessing_log.to_csv(OUTPUT_DIR / 'world_bank_group1_preprocessing_log.csv', index=False)

missing_after = clean_df[ANALYSIS_COLUMNS].isna().sum()
if int(missing_after.sum()) != 0:
    raise ValueError(f'Cleaning failed: missing values remain: {missing_after.to_dict()}')

treatment_summary = pd.DataFrame({
    'indicator': ['GDP per capita', 'GDP growth', 'Life expectancy', 'Secondary enrolment'],
    'missing_before': [int(original_missing_mask[c].sum()) for c in ANALYSIS_COLUMNS],
    'missing_after': [int(clean_df[c].isna().sum()) for c in ANALYSIS_COLUMNS],
    'imputed_values': [int(clean_df[f'{c}_was_imputed'].sum()) for c in ANALYSIS_COLUMNS],
    'outlier_candidates': [int(clean_df[f'{c}_outlier_candidate'].sum()) for c in ANALYSIS_COLUMNS],
})
treatment_summary.to_csv(OUTPUT_DIR / 'world_bank_group1_treatment_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
x = np.arange(len(treatment_summary))
axes[0].bar(x - 0.18, treatment_summary['missing_before'], width=0.36, color=SU_RED, label='Before')
axes[0].bar(x + 0.18, treatment_summary['missing_after'], width=0.36, color=SU_BLUE, label='After')
axes[0].set_xticks(x, treatment_summary['indicator'], rotation=20, ha='right')
axes[0].set_ylabel('Missing cells')
axes[0].set_title('Missing values: before and after', weight='bold', color=SU_BLUE)
axes[0].legend(frameon=False)
axes[1].bar(treatment_summary['indicator'], treatment_summary['outlier_candidates'], color=SU_GOLD)
axes[1].set_xticklabels(treatment_summary['indicator'], rotation=20, ha='right')
axes[1].set_ylabel('IQR candidates reviewed')
axes[1].set_title('Outliers were reviewed, not blindly removed', weight='bold', color=SU_BLUE)
fig.suptitle('Data treatment summary', weight='bold', color=SU_BLUE, y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '00b_data_treatment_summary.png', dpi=300, bbox_inches='tight')
plt.show()

display(preprocessing_log)
display(treatment_summary)
print('Fully cleaned data shape:', clean_df.shape)
print('Missing analytical values after cleaning:', int(clean_df[ANALYSIS_COLUMNS].isna().sum().sum()))
print('Use this submission file:', complete_csv_path)


# Part C - Exploratory data analysis

## Step 16 - Descriptive statistics and complete 2010-2024 endpoint changes

The final dataset contains all 120 country-year rows with complete analytical values. Every estimated value remains traceable through a `was_imputed` flag and a method column. Endpoint comparisons now use 2010 and 2024 for all eight countries, but results with many imputed values must still be presented cautiously.

> **Presentation explanation:** We created one complete regional panel, then used the same start and end years for every country.


In [ ]:
descriptive_statistics = clean_df[ANALYSIS_COLUMNS].describe().T
descriptive_statistics['missing_count_after_cleaning'] = clean_df[ANALYSIS_COLUMNS].isna().sum()
descriptive_statistics['available_pct_after_cleaning'] = (
    100 * clean_df[ANALYSIS_COLUMNS].notna().mean()
).round(1)
descriptive_statistics['imputed_count'] = [
    int(clean_df[f'{column}_was_imputed'].sum()) for column in ANALYSIS_COLUMNS
]
display(descriptive_statistics)

def endpoint_change_table(data, value_column):
    output_columns = [
        'country_code', 'country', 'variable', 'start_year', 'end_year',
        'span_years', 'start_value', 'end_value', 'absolute_change',
        'percent_change', 'annualised_percent',
        'annualised_absolute_change', 'strict_2010_2024'
    ]
    results = []
    for country_code, country_name in COUNTRIES.items():
        observed = (
            data.loc[data['country_code'].eq(country_code), ['year', value_column]]
            .dropna()
            .drop_duplicates('year')
            .sort_values('year')
        )
        if len(observed) < 2:
            continue
        first = observed.iloc[0]
        last = observed.iloc[-1]
        span_years = int(last['year'] - first['year'])
        if span_years <= 0:
            continue
        start_value = float(first[value_column])
        end_value = float(last[value_column])
        absolute_change = end_value - start_value
        percent_change = (100 * absolute_change / start_value) if start_value != 0 else np.nan
        annualised_percent = (
            100 * ((end_value / start_value) ** (1 / span_years) - 1)
            if start_value > 0 and end_value > 0 else np.nan
        )
        results.append({
            'country_code': country_code,
            'country': country_name,
            'variable': value_column,
            'start_year': int(first['year']),
            'end_year': int(last['year']),
            'span_years': span_years,
            'start_value': start_value,
            'end_value': end_value,
            'absolute_change': absolute_change,
            'percent_change': percent_change,
            'annualised_percent': annualised_percent,
            'annualised_absolute_change': absolute_change / span_years,
            'strict_2010_2024': bool(
                first['year'] == START_YEAR and last['year'] == END_YEAR
            ),
        })
    return pd.DataFrame(results, columns=output_columns)

def select_preferred_period(change_table, ranking_column, ascending=False):
    selected = change_table.loc[change_table['strict_2010_2024']].copy()
    basis = (
        f'Completed 2010-2024 series: {len(selected)} countries included; '
        'estimated cells are explicitly flagged in the clean CSV'
    )
    return selected.sort_values(ranking_column, ascending=ascending), basis

gdp_change = endpoint_change_table(clean_df, GDP_LEVEL)
life_change = endpoint_change_table(clean_df, LIFE)
education_change = endpoint_change_table(clean_df, EDUCATION)
observed_education_change = endpoint_change_table(analysis_raw, EDUCATION)

gdp_ranked, gdp_comparison_basis = select_preferred_period(gdp_change, 'percent_change')
education_ranked, education_comparison_basis = select_preferred_period(
    education_change, 'absolute_change'
)

print('GDP comparison basis:', gdp_comparison_basis)
display(gdp_ranked.round(3))
print('Education comparison basis:', education_comparison_basis)
display(education_ranked.round(3))


## Step 17 - Compare GDP and life-expectancy improvement rates

GDP and life expectancy have different units, so the rate comparison uses annualised percentage change for both. A value of zero in `rate_gap_percentage_points` would indicate identical annualised rates. This is a descriptive comparison, not proof that GDP caused health improvement.

> **Presentation explanation:** We converted GDP and life expectancy changes to average yearly percentages so they could be compared.

In [ ]:
rate_comparison = (
    gdp_change[[
        'country_code', 'country', 'annualised_percent',
        'start_year', 'end_year', 'span_years'
    ]]
    .rename(columns={
        'annualised_percent': 'gdp_annualised_pct',
        'start_year': 'gdp_start_year',
        'end_year': 'gdp_end_year',
        'span_years': 'gdp_span_years',
    })
    .merge(
        life_change[[
            'country_code', 'annualised_percent',
            'start_year', 'end_year', 'span_years'
        ]].rename(columns={
            'annualised_percent': 'life_expectancy_annualised_pct',
            'start_year': 'life_start_year',
            'end_year': 'life_end_year',
            'span_years': 'life_span_years',
        }),
        on='country_code', how='inner'
    )
)
rate_comparison['rate_gap_percentage_points'] = (
    rate_comparison['gdp_annualised_pct']
    - rate_comparison['life_expectancy_annualised_pct']
)
rate_comparison['same_rate_within_0_1_pp'] = (
    rate_comparison['rate_gap_percentage_points'].abs() <= 0.1
)
rate_comparison = rate_comparison.sort_values(
    'rate_gap_percentage_points', ascending=False
)
full_period_rate_comparison = rate_comparison.loc[
    rate_comparison['gdp_start_year'].eq(START_YEAR)
    & rate_comparison['gdp_end_year'].eq(END_YEAR)
    & rate_comparison['life_start_year'].eq(START_YEAR)
    & rate_comparison['life_end_year'].eq(END_YEAR)
].copy()
display(rate_comparison.round(3))

## Step 18 - Construct a transparent balanced-improvement score

This is an analyst-designed index, not an official World Bank measure. Economy uses annualised real-GDP-per-capita percentage growth, health uses annualised life-expectancy percentage growth, and education uses annualised percentage-point change in gross enrolment. Each component is min-max normalised to 0-100 among countries with at least five observed years in all three areas. The final score equals mean progress minus the standard deviation across the three components, rewarding both strength and balance.

> **Presentation explanation:** We combined economy, health, and education scores and penalised countries with very uneven progress.

In [ ]:
balanced = (
    gdp_change[[
        'country_code', 'country', 'annualised_percent', 'span_years'
    ]].rename(columns={
        'annualised_percent': 'economy_rate',
        'span_years': 'economy_span',
    })
    .merge(
        life_change[['country_code', 'annualised_percent', 'span_years']].rename(columns={
            'annualised_percent': 'health_rate',
            'span_years': 'health_span',
        }), on='country_code', how='inner'
    )
    .merge(
        education_change[[
            'country_code', 'annualised_absolute_change', 'span_years'
        ]].rename(columns={
            'annualised_absolute_change': 'education_rate',
            'span_years': 'education_span',
        }), on='country_code', how='inner'
    )
)
balanced = balanced.loc[
    (balanced[['economy_span', 'health_span', 'education_span']] >= 5).all(axis=1)
].copy()

def minmax_100(series):
    result = pd.Series(np.nan, index=series.index, dtype=float)
    valid = series.dropna()
    if valid.empty:
        return result
    spread = valid.max() - valid.min()
    if spread == 0:
        result.loc[valid.index] = 50.0
    else:
        result.loc[valid.index] = 100 * (valid - valid.min()) / spread
    return result

for raw_column, score_column in [
    ('economy_rate', 'economy_score'),
    ('health_rate', 'health_score'),
    ('education_rate', 'education_score'),
]:
    balanced[score_column] = minmax_100(balanced[raw_column])

component_scores = ['economy_score', 'health_score', 'education_score']
balanced['mean_progress_score'] = balanced[component_scores].mean(axis=1)
balanced['imbalance_penalty'] = balanced[component_scores].std(axis=1, ddof=0)
balanced['balanced_improvement_score'] = (
    balanced['mean_progress_score'] - balanced['imbalance_penalty']
)
balanced = balanced.sort_values('balanced_improvement_score', ascending=False)
display(balanced.round(2))

# Part D - Statistical analysis

The notebook applies three statistical techniques:

1. **Trend analysis:** country-specific linear slopes over time.
2. **Correlation analysis:** Pearson linear correlation and Spearman rank correlation between real GDP per capita and life expectancy.
3. **Multiple regression:** life expectancy modelled against log real GDP per capita, a time trend, and country controls, using HC3 robust standard errors.

Significance is assessed at alpha = 0.05, but p-values should be interpreted with effect sizes, sample size, missingness, and the observational nature of the data.

## Step 19 - Statistical technique 1: country-level linear trend analysis

> **Presentation explanation:** We fitted a straight trend line for each country and indicator to measure the average yearly direction.

In [ ]:
trend_rows = []
trend_metrics = {
    GDP_LEVEL: 'GDP per capita (constant 2015 US$)',
    LIFE: 'Life expectancy (years)',
    EDUCATION: 'Secondary enrolment (% gross)',
}

for value_column, label in trend_metrics.items():
    for country_code, country_name in COUNTRIES.items():
        sample = clean_df.loc[
            clean_df['country_code'].eq(country_code), ['year', value_column]
        ].dropna()
        if len(sample) < 3 or sample['year'].nunique() < 3:
            continue
        result = stats.linregress(
            sample['year'].astype(float), sample[value_column].astype(float)
        )
        trend_rows.append({
            'country_code': country_code,
            'country': country_name,
            'variable': value_column,
            'indicator_name': label,
            'n_observations': len(sample),
            'slope_per_year': result.slope,
            'intercept': result.intercept,
            'r_squared': result.rvalue ** 2,
            'p_value_for_slope': result.pvalue,
            'significant_at_0_05': result.pvalue < 0.05,
        })

trend_result_columns = [
    'country_code', 'country', 'variable', 'indicator_name',
    'n_observations', 'slope_per_year', 'intercept', 'r_squared',
    'p_value_for_slope', 'significant_at_0_05'
]
trend_results = pd.DataFrame(trend_rows, columns=trend_result_columns)
if not trend_results.empty:
    trend_results = trend_results.sort_values(
        ['variable', 'slope_per_year'], ascending=[True, False]
    )
trend_results.to_csv(OUTPUT_DIR / 'world_bank_group1_trend_analysis.csv', index=False)
display(trend_results.round(4))

## Step 20 - Statistical technique 2: Pearson and Spearman correlation

Pearson measures linear association; Spearman measures rank association and is less sensitive to extreme values. We run both tests on the completed 120-row panel and repeat them on reported-only pairs as a sensitivity check. Correlation does not establish causation.

> **Presentation explanation:** We tested the completed dataset, then checked whether the answer changed when we used only reported values.


In [ ]:
correlation_sample = clean_df[[
    'country_code', 'country', 'year', GDP_LEVEL, LIFE
]].copy()
observed_correlation_sample = analysis_raw[[
    'country_code', 'country', 'year', GDP_LEVEL, LIFE
]].dropna().copy()

correlation_rows = []
for sample_name, sample in [
    ('completed_panel', correlation_sample),
    ('reported_only', observed_correlation_sample),
]:
    if (
        len(sample) >= 3
        and sample[GDP_LEVEL].nunique() > 1
        and sample[LIFE].nunique() > 1
    ):
        pearson = stats.pearsonr(sample[GDP_LEVEL], sample[LIFE])
        spearman = stats.spearmanr(sample[GDP_LEVEL], sample[LIFE])
        correlation_rows.extend([
            {'method': f'Pearson_{sample_name}', 'sample': sample_name, 'coefficient': pearson.statistic, 'p_value': pearson.pvalue, 'n': len(sample)},
            {'method': f'Spearman_{sample_name}', 'sample': sample_name, 'coefficient': spearman.statistic, 'p_value': spearman.pvalue, 'n': len(sample)},
        ])

correlation_results = pd.DataFrame(correlation_rows)
correlation_results['significant_at_0_05'] = correlation_results['p_value'] < 0.05
correlation_results.to_csv(OUTPUT_DIR / 'world_bank_group1_correlation_results.csv', index=False)
display(correlation_results.round(5))


## Step 21 - Statistical technique 3: multiple regression with country controls

Model: `life expectancy ~ log(real GDP per capita) + calendar-year trend + country indicators`. Country indicators control for fixed differences between countries, while HC3 robust standard errors reduce sensitivity to unequal variance. We fit the model to the completed panel and repeat it on reported-only GDP observations. Different conclusions across these models indicate sensitivity, not a stable causal effect.

> **Presentation explanation:** We controlled for country and year, then repeated the model without imputed GDP to see whether the conclusion stayed the same.


In [ ]:
regression_sample = clean_df[[
    'country_code', 'year', LIFE, 'log_gdp_per_capita'
]].copy()
regression_sample['year_centered'] = regression_sample['year'] - START_YEAR

observed_regression_sample = analysis_raw[[
    'country_code', 'year', LIFE, GDP_LEVEL
]].dropna().copy()
observed_regression_sample['log_gdp_per_capita'] = np.log(observed_regression_sample[GDP_LEVEL])
observed_regression_sample['year_centered'] = observed_regression_sample['year'] - START_YEAR

def fit_controlled_regression(sample, sample_name):
    if len(sample) < 20 or sample['country_code'].nunique() < 2:
        return None, pd.DataFrame()
    model = smf.ols(
        formula=f'{LIFE} ~ log_gdp_per_capita + year_centered + C(country_code)',
        data=sample,
    ).fit(cov_type='HC3')
    coefficients = pd.DataFrame({
        'sample': sample_name,
        'term': model.params.index,
        'coefficient': model.params.values,
        'robust_standard_error': model.bse.values,
        'p_value': model.pvalues.values,
        'ci_95_lower': model.conf_int()[0].values,
        'ci_95_upper': model.conf_int()[1].values,
    })
    coefficients['significant_at_0_05'] = coefficients['p_value'] < 0.05
    return model, coefficients

regression_model, regression_coefficients = fit_controlled_regression(
    regression_sample, 'completed_panel'
)
observed_regression_model, observed_regression_coefficients = fit_controlled_regression(
    observed_regression_sample, 'reported_only'
)

if regression_model is not None:
    with (OUTPUT_DIR / 'world_bank_group1_regression_summary.txt').open('w') as file:
        file.write(regression_model.summary().as_text())
    print('Completed-panel model')
    print(regression_model.summary())
if observed_regression_model is not None:
    with (OUTPUT_DIR / 'world_bank_group1_regression_reported_only_summary.txt').open('w') as file:
        file.write(observed_regression_model.summary().as_text())
    print('Reported-only sensitivity model')
    print(observed_regression_model.summary())

regression_coefficients.to_csv(
    OUTPUT_DIR / 'world_bank_group1_regression_coefficients.csv', index=False
)
observed_regression_coefficients.to_csv(
    OUTPUT_DIR / 'world_bank_group1_regression_reported_only_coefficients.csv', index=False
)
display(pd.concat([
    regression_coefficients.loc[regression_coefficients['term'].isin(['log_gdp_per_capita', 'year_centered'])],
    observed_regression_coefficients.loc[observed_regression_coefficients['term'].isin(['log_gdp_per_capita', 'year_centered'])],
]).round(5))


# Part E - Direct answers, visualisations, and recommendations

## Visualisation plan

| Chart | Question | Why it was selected | Good practice used |
|---|---|---|---|
| 1. GDP growth ranking | Q1 | Sorted bars show the highest growth clearly | The completed 2010-2024 endpoints and imputation caveats are shown |
| 2. Indexed GDP and life expectancy | Q2 | Both measures start at 100, so their rates can be compared fairly | Same scale; no confusing double axis |
| 3. School-enrolment ranking | Q3 | Sorted bars make the largest change easy to see | Percentage-point units and reported-versus-imputed years are shown |
| 4. GDP-life expectancy scatterplot | Q4 | Dots and a fitted line show the relationship and spread | Country colours, sample size, correlation, p-value, and controlled result |
| 5. Balanced-improvement heatmap | Q5 | Shows the final score and all three parts | Common scale and values printed in every cell; the penalty can make the final score negative |
| Additional GDP trend chart | Extra context | Shows the yearly path behind the Q1 ranking | Separate country panels avoid overlapping lines |

All charts use clear units, accessible colours, light gridlines, and enough space above the plot so headings do not cover explanations.

## Step 22 - Question 1: Which country had the highest GDP growth per person?

**How we solved it:** We calculated the percentage change in inflation-adjusted GDP per person from 2010 to 2024 for every country, using the completed series, then ranked the countries.

**Why this chart:** Sorted horizontal bars make the leading country easy to identify.

> **What to say in the presentation:** We used the same 2010-2024 period for all countries and kept imputation flags beside the completed dataset.


In [ ]:
gdp_bar_data = gdp_ranked.sort_values('percent_change').copy()
if gdp_bar_data.empty:
    print('Insufficient GDP endpoint observations for Visualisation 2.')
else:
    gdp_leader = gdp_ranked.iloc[0]
    display(Markdown(
        f"### Direct answer to Question 1\n"
        f"**{gdp_leader['country']}** recorded the highest comparable growth in real GDP "
        f"per person: **{gdp_leader['percent_change']:.1f}%** between "
        f"{int(gdp_leader['start_year'])} and {int(gdp_leader['end_year'])}.\n\n"
        f"*Comparison used: {gdp_comparison_basis}.*"
    ))
    fig, ax = plt.subplots(figsize=(10, 6))
    colours = [country_palette[code] for code in gdp_bar_data['country_code']]
    bars = ax.barh(gdp_bar_data['country'], gdp_bar_data['percent_change'], color=colours)
    ax.axvline(0, color='0.35', linewidth=1)
    ax.set_xlabel('Total change in the completed 2010-2024 series (%)')
    ax.set_ylabel('')
    ax.grid(axis='y', visible=False)
    for bar, row in zip(bars, gdp_bar_data.itertuples()):
        value = row.percent_change
        offset = 3
        alignment = 'left'
        ax.annotate(
            f'{value:.1f}% ({row.start_year}-{row.end_year})',
            xy=(value, bar.get_y() + bar.get_height() / 2),
            xytext=(offset, 0), textcoords='offset points',
            va='center', ha=alignment, fontsize=9
        )
    add_chart_heading(
        fig, 'Real GDP per person growth ranking',
        subtitle=gdp_comparison_basis, top=0.82
    )
    fig.savefig(FIGURE_DIR / '02_gdp_growth_ranking.png', dpi=300, bbox_inches='tight')
    plt.show()

## Step 23 - Question 2: Did life expectancy improve as fast as GDP per person?

**How we solved it:** We converted both measures to an average percentage change per year. We then checked whether the two rates were within 0.1 percentage points for each country.

**Why this chart:** Starting both lines at 100 lets us compare their movement without mixing dollars and years on two axes.

> **What to say in the presentation:** We put GDP and life expectancy on the same starting scale and compared the rates for all eight countries.


In [ ]:
indexed_rows = []
for country_code, country_name in COUNTRIES.items():
    country_data = clean_df.loc[clean_df['country_code'].eq(country_code)].sort_values('year')
    for value_column, measure_label in [
        (GDP_LEVEL, 'Real GDP per capita'),
        (LIFE, 'Life expectancy'),
    ]:
        complete = country_data[['year', value_column]].dropna()
        if complete.empty or complete.iloc[0][value_column] == 0:
            continue
        base_value = complete.iloc[0][value_column]
        base_year = int(complete.iloc[0]['year'])
        for row in complete.itertuples(index=False):
            indexed_rows.append({
                'country_code': country_code,
                'country': country_name,
                'year': int(row.year),
                'measure': measure_label,
                'index_value': 100 * getattr(row, value_column) / base_value,
                'base_year': base_year,
            })
indexed_data = pd.DataFrame(indexed_rows)

if not full_period_rate_comparison.empty:
    same_rate_count = int(full_period_rate_comparison['same_rate_within_0_1_pp'].sum())
    comparison_count = len(full_period_rate_comparison)
    if same_rate_count == 0:
        rate_answer = 'No country had the two measures improve at almost the same average yearly rate.'
    else:
        rate_answer = (
            f'{same_rate_count} of {comparison_count} countries had the two measures improve '
            'at almost the same average yearly rate.'
        )
    widest_gap = full_period_rate_comparison.iloc[
        full_period_rate_comparison['rate_gap_percentage_points'].abs().argmax()
    ]
    gdp_imputed_countries = int(clean_df.groupby('country_code')[f'{GDP_LEVEL}_was_imputed'].any().sum())
    display(Markdown(
        '### Direct answer to Question 2\n'
        f'**{rate_answer}** The largest full-period gap was in '
        f'**{widest_gap["country"]}**: GDP per person changed by '
        f'{widest_gap["gdp_annualised_pct"]:.2f}% per year, while life expectancy changed '
        f'by {widest_gap["life_expectancy_annualised_pct"]:.2f}% per year. '
        f'All {comparison_count} countries were included; {gdp_imputed_countries} country required GDP imputation.'
    ))
    display(full_period_rate_comparison[[
        'country', 'gdp_annualised_pct', 'life_expectancy_annualised_pct',
        'rate_gap_percentage_points', 'same_rate_within_0_1_pp'
    ]].round(3))

if indexed_data.empty:
    print('Insufficient data for indexed comparison.')
else:
    chart3 = sns.relplot(
        data=indexed_data, x='year', y='index_value',
        hue='measure', style='measure',
        palette={'Real GDP per capita': SU_BLUE, 'Life expectancy': SU_RED},
        dashes={'Real GDP per capita': '', 'Life expectancy': (4, 2)},
        col='country', col_wrap=4, kind='line', marker='o',
        linewidth=2, height=3.1, aspect=1.15,
        facet_kws={'sharex': True, 'sharey': True},
    )
    chart3.map(plt.axhline, y=100, color='0.65', linewidth=0.8, linestyle=':')
    chart3.set_titles('{col_name}')
    chart3.set_axis_labels('Year', 'Index (2010 = 100)')
    chart3.figure.suptitle(
        'Relative improvement in real GDP per capita and life expectancy',
        y=0.995, weight='bold', color=SU_BLUE
    )
    chart3.figure.subplots_adjust(top=0.84, hspace=0.38, wspace=0.18)
    chart3.figure.savefig(
        FIGURE_DIR / '03_indexed_gdp_vs_life_expectancy.png',
        dpi=300, bbox_inches='tight'
    )
    plt.show()


## Step 24 - Question 3: Which country improved secondary school enrolment the most?

**How we solved it:** We completed the missing annual values, compared 2010 with 2024 for every country, and then checked how many reported values supported the leading result.

**Why this chart:** A sorted bar chart shows the largest estimated increase clearly. The answer also states the reported-versus-imputed evidence behind it.

> **What to say in the presentation:** The completed dataset makes the periods comparable, but the ranking is less certain where most values were estimated.


In [ ]:
education_bar_data = education_ranked.sort_values('absolute_change').copy()
if education_bar_data.empty:
    print('Insufficient education endpoint observations for Visualisation 4.')
else:
    education_leader = education_ranked.iloc[0]
    original_full_coverage_count = int(observed_education_change['strict_2010_2024'].sum())
    leader_outlier_flag = bool((
        outlier_detail['country_code'].eq(education_leader['country_code'])
        & outlier_detail['variable'].eq(EDUCATION)
    ).any()) if not outlier_detail.empty else False
    leader_observation_count = int(analysis_raw.loc[
        analysis_raw['country_code'].eq(education_leader['country_code']), EDUCATION
    ].notna().sum())
    leader_imputed_count = int(clean_df.loc[
        clean_df['country_code'].eq(education_leader['country_code']),
        f'{EDUCATION}_was_imputed'
    ].sum())
    quality_note = (
        f' The completed 15-year series contains {leader_observation_count} reported and '
        f'{leader_imputed_count} imputed values'
        + ('; one reported value was reviewed as an IQR candidate.' if leader_outlier_flag else '.')
    )
    display(Markdown(
        f"### Direct answer to Question 3\n"
        f"After completing the annual series, **{education_leader['country']}** had the "
        f"largest estimated increase: **{education_leader['absolute_change']:.1f} percentage points** "
        f"between {int(education_leader['start_year'])} and {int(education_leader['end_year'])}. "
        f"Before imputation, only {original_full_coverage_count} country had both exact endpoint values."
        f"{quality_note} The result should therefore be treated as provisional."
    ))
    fig, ax = plt.subplots(figsize=(10, 6))
    colours = [country_palette[code] for code in education_bar_data['country_code']]
    bars = ax.barh(
        education_bar_data['country'], education_bar_data['absolute_change'], color=colours
    )
    ax.axvline(0, color='0.35', linewidth=1)
    ax.set_xlabel('Change in completed 2010-2024 series (percentage points)')
    ax.set_ylabel('')
    ax.grid(axis='y', visible=False)
    for bar, row in zip(bars, education_bar_data.itertuples()):
        value = row.absolute_change
        label_x = value if value >= 0 else 0
        ax.annotate(
            f'{value:.1f} pp ({row.start_year}-{row.end_year})',
            xy=(label_x, bar.get_y() + bar.get_height() / 2),
            xytext=(3, 0), textcoords='offset points',
            va='center', ha='left', fontsize=9
        )
    add_chart_heading(
        fig, 'Estimated improvement in secondary school enrolment',
        subtitle=education_comparison_basis, top=0.82
    )
    fig.savefig(
        FIGURE_DIR / '04_secondary_enrolment_improvement.png',
        dpi=300, bbox_inches='tight'
    )
    plt.show()


## Step 25 - Question 4: Is GDP per person related to life expectancy?

**How we solved it:** We compared correlation and controlled regression results for both the completed panel and reported-only observations. If the conclusion changes, the relationship is not robust.

**Why this chart:** Each dot is one country-year in the completed dataset. The fitted line shows the overall direction, while the annotation reports the sensitivity results.

> **What to say in the presentation:** The answer changes with imputation and controls, so GDP should not be presented as an automatic cause of longer life.


In [ ]:
scatter_data = correlation_sample.copy()
scatter_data['log_gdp_per_capita'] = np.log(scatter_data[GDP_LEVEL])

if scatter_data.empty:
    print('No paired GDP and life-expectancy observations for Visualisation 5.')
else:
    completed_pearson = correlation_results.loc[
        correlation_results['method'].eq('Pearson_completed_panel')
    ].iloc[0]
    reported_pearson = correlation_results.loc[
        correlation_results['method'].eq('Pearson_reported_only')
    ].iloc[0]
    completed_control = regression_coefficients.loc[
        regression_coefficients['term'].eq('log_gdp_per_capita')
    ].iloc[0]
    reported_control = observed_regression_coefficients.loc[
        observed_regression_coefficients['term'].eq('log_gdp_per_capita')
    ].iloc[0]
    relationship_answer = (
        'The relationship is not robust. The completed-panel Pearson test is not significant, '
        'the completed controlled model is significant, and the reported-only controlled model '
        'is not significant. The conclusion depends on preprocessing and model choice.'
    )
    display(Markdown(
        '### Direct answer to Question 4\n'
        f'**{relationship_answer}** Completed-panel Pearson r={completed_pearson["coefficient"]:.3f}, '
        f'p={completed_pearson["p_value"]:.3g}, n={int(completed_pearson["n"])}. '
        f'Completed controlled GDP p={completed_control["p_value"]:.3f}; '
        f'reported-only controlled GDP p={reported_control["p_value"]:.3f}. '
        'This analysis shows association, not cause.'
    ))
    fig, ax = plt.subplots(figsize=(10, 6.5))
    sns.scatterplot(
        data=scatter_data, x='log_gdp_per_capita', y=LIFE,
        hue='country_code', palette=country_palette,
        s=65, alpha=0.78, edgecolor='white', linewidth=0.4, ax=ax
    )
    sns.regplot(
        data=scatter_data, x='log_gdp_per_capita', y=LIFE,
        scatter=False, color=SU_RED, ci=95,
        line_kws={'linewidth': 2}, ax=ax
    )
    ax.set_xlabel('Natural log of GDP per capita (constant 2015 US$)')
    ax.set_ylabel('Life expectancy at birth (years)')
    ax.text(
        0.02, 0.98,
        f'Completed Pearson r={completed_pearson.coefficient:.3f}, p={completed_pearson.p_value:.3g}, n={int(completed_pearson.n)}\n'
        f'Controlled GDP p: completed={completed_control.p_value:.3f}; reported-only={reported_control.p_value:.3f}',
        transform=ax.transAxes, va='top',
        bbox={'boxstyle': 'round,pad=0.3', 'facecolor': SU_LIGHT_BLUE, 'alpha': 0.9, 'edgecolor': '#D7D9E5'}
    )
    ax.legend(title='Country code', bbox_to_anchor=(1.02, 1), loc='upper left')
    add_chart_heading(fig, 'GDP per person and life expectancy', top=0.91)
    fig.savefig(
        FIGURE_DIR / '05_gdp_life_expectancy_relationship.png',
        dpi=300, bbox_inches='tight'
    )
    plt.show()


## Step 26 - Question 5: Which country improved most evenly across all three areas?

**How we solved it:** We gave economy, health, and education comparable scores from 0 to 100. We averaged them and subtracted a penalty when one area was much weaker than the others.

**Why this chart:** The heatmap shows the final score and all three parts, so a weak area cannot be hidden.

> **What to say in the presentation:** The winner was not simply the country with the fastest GDP growth; it was the country with the strongest and most even progress across all three areas.

In [ ]:
if balanced.empty:
    print('Insufficient overlapping endpoint information for the balanced score.')
else:
    balance_leader = balanced.iloc[0]
    display(Markdown(
        f"### Direct answer to Question 5\n"
        f"**{balance_leader['country']}** had the highest balance-adjusted score: "
        f"**{balance_leader['balanced_improvement_score']:.1f}** (higher is better). Its component "
        f"scores were economy {balance_leader['economy_score']:.1f}, health "
        f"{balance_leader['health_score']:.1f}, and education "
        f"{balance_leader['education_score']:.1f}. This is a dashboard score, not an official "
        f"World Bank index."
    ))
    balance_matrix = (
        balanced.set_index('country')[[
            'economy_score', 'health_score', 'education_score',
            'balanced_improvement_score'
        ]]
        .rename(columns={
            'economy_score': 'Economy',
            'health_score': 'Health',
            'education_score': 'Education',
            'balanced_improvement_score': 'Balanced score',
        })
    )
    fig, ax = plt.subplots(figsize=(9.5, max(4.5, 0.55 * len(balance_matrix) + 2)))
    heatmap_min = min(0, float(np.nanmin(balance_matrix.to_numpy())))
    sns.heatmap(
        balance_matrix, annot=True, fmt='.1f', cmap=su_diverging,
        vmin=heatmap_min, vmax=100, linewidths=0.5,
        cbar_kws={'label': 'Score (components are 0-100)'}, ax=ax
    )
    ax.set_xlabel('')
    ax.set_ylabel('')
    add_chart_heading(
        fig, 'Balanced improvement across economy, health, and education', top=0.91
    )
    fig.savefig(
        FIGURE_DIR / '06_balanced_improvement_heatmap.png',
        dpi=300, bbox_inches='tight'
    )
    plt.show()


## Step 27 - Question 6: What should the East African Community do?

**How we solved it:** We created recommendations only when a statistic or result from this notebook could be named beside it. The table below shows the action, the supporting evidence, and why the evidence matters.

> **What to say in the presentation:** Every recommendation comes directly from a result in our analysis; we did not add unsupported opinions.

In [ ]:
policy_rows = []

if not full_period_rate_comparison.empty:
    same_rate_count = int(full_period_rate_comparison['same_rate_within_0_1_pp'].sum())
    largest_positive_gap = full_period_rate_comparison.sort_values(
        'rate_gap_percentage_points', ascending=False
    ).iloc[0]
    policy_rows.append({
        'recommendation': 'Link economic plans to specific health targets and budgets.',
        'evidence_from_analysis': (
            f'{same_rate_count} of {len(full_period_rate_comparison)} countries in the completed '
            f'2010-2024 panel had GDP per person and life expectancy improve at almost the same '
            f'yearly rate. {largest_positive_gap["country"]} had the largest positive gap: '
            f'{largest_positive_gap["gdp_annualised_pct"]:.2f}% GDP growth per year versus '
            f'{largest_positive_gap["life_expectancy_annualised_pct"]:.2f}% life-expectancy growth per year.'
        ),
        'why_this_supports_the_action': 'Economic growth was not automatically matched by equally fast health improvement.',
    })

education_completeness_pct = 100 * analysis_raw[EDUCATION].notna().mean()
strict_education_count = int(observed_education_change['strict_2010_2024'].sum())
policy_rows.append({
    'recommendation': 'Create a common EAC annual reporting standard for secondary education.',
    'evidence_from_analysis': (
        f'Only {education_completeness_pct:.1f}% of expected secondary-enrolment values were '
        f'reported, and only {strict_education_count} country had both reported 2010 and 2024 values.'
    ),
    'why_this_supports_the_action': 'Large reporting gaps make country rankings uncertain and weaken regional planning.',
})

if not education_ranked.empty:
    education_leader = education_ranked.iloc[0]
    reported_count = int(analysis_raw.loc[
        analysis_raw['country_code'].eq(education_leader['country_code']), EDUCATION
    ].notna().sum())
    policy_rows.append({
        'recommendation': 'Validate sparse enrolment series before copying another country’s approach.',
        'evidence_from_analysis': (
            f'{education_leader["country"]} had the largest completed-series increase '
            f'({education_leader["absolute_change"]:.1f} percentage points from 2010 to 2024), '
            f'but only {reported_count} of 15 annual values were reported.'
        ),
        'why_this_supports_the_action': 'A large imputation-dependent result should be verified before it becomes a regional model.',
    })

if not balanced.empty:
    balance_leader = balanced.iloc[0]
    component_values = {
        'economy': balance_leader['economy_score'],
        'health': balance_leader['health_score'],
        'education': balance_leader['education_score'],
    }
    weakest_area = min(component_values, key=component_values.get)
    policy_rows.append({
        'recommendation': 'Use a regional scorecard that tracks economy, health, and education together.',
        'evidence_from_analysis': (
            f'{balance_leader["country"]} ranked first on the balanced score '
            f'({balance_leader["balanced_improvement_score"]:.1f}), but its weakest component was '
            f'{weakest_area} ({component_values[weakest_area]:.1f}/100).'
        ),
        'why_this_supports_the_action': 'Even the most balanced country still had a weak area, so GDP alone gives an incomplete view.',
    })

completed_pearson = correlation_results.loc[
    correlation_results['method'].eq('Pearson_completed_panel')
].iloc[0]
completed_control = regression_coefficients.loc[
    regression_coefficients['term'].eq('log_gdp_per_capita')
].iloc[0]
reported_control = observed_regression_coefficients.loc[
    observed_regression_coefficients['term'].eq('log_gdp_per_capita')
].iloc[0]
policy_rows.append({
    'recommendation': 'Do not assume that raising GDP alone will raise life expectancy.',
    'evidence_from_analysis': (
        f'The completed-panel Pearson test was not significant (r={completed_pearson["coefficient"]:.3f}, '
        f'p={completed_pearson["p_value"]:.3g}); the completed controlled GDP term was significant '
        f'(p={completed_control["p_value"]:.3f}), but the reported-only controlled term was not '
        f'(p={reported_control["p_value"]:.3f}).'
    ),
    'why_this_supports_the_action': 'The conclusion changes with imputation and model choice, so the relationship is not robust.',
})

policy_recommendations = pd.DataFrame(policy_rows)
policy_recommendations.to_csv(
    OUTPUT_DIR / 'world_bank_group1_policy_recommendations.csv', index=False
)
display(Markdown('### Direct answer to Question 6'))
for number, row in enumerate(policy_recommendations.itertuples(index=False), start=1):
    display(Markdown(
        f'**{number}. {row.recommendation}**\n\n'
        f'- **Evidence:** {row.evidence_from_analysis}\n'
        f'- **Why it supports the recommendation:** {row.why_this_supports_the_action}'
    ))
display(policy_recommendations)


## Additional visualisation - GDP per person trends

**How we solved it:** We drew a separate line chart for each country to show the yearly path behind the start-to-end ranking.

**Why it appears here:** This chart gives extra context after the six answers, as requested.

> **What to say in the presentation:** The ranking gives the overall change, while these lines show whether progress was steady or interrupted.

In [ ]:
gdp_plot_data = clean_df.dropna(subset=[GDP_LEVEL]).copy()
if gdp_plot_data.empty:
    print('No GDP-per-capita observations available for Visualisation 1.')
else:
    chart1 = sns.relplot(
        data=gdp_plot_data, x='year', y=GDP_LEVEL,
        hue='country_code', palette=country_palette,
        col='country', col_wrap=4, kind='line',
        marker='o', linewidth=2, height=3.1, aspect=1.15,
        facet_kws={'sharex': True, 'sharey': True}, legend=False,
    )
    chart1.set_titles('{col_name}')
    chart1.set_axis_labels('Year', 'Constant 2015 US$ per person')
    chart1.figure.suptitle(
        'Real GDP per capita trends in EAC Partner States, 2010-2024',
        y=0.995, weight='bold'
    )
    chart1.figure.subplots_adjust(top=0.86, hspace=0.38, wspace=0.18)
    chart1.figure.savefig(
        FIGURE_DIR / '01_gdp_per_capita_trends.png', dpi=300, bbox_inches='tight'
    )
    plt.show()

# Part F - Export the complete analysis, answers, and recommendations

## Step 28 - Save all tables and create a research-question summary

The notebook now saves the analysis tables, direct answer summary, and evidence-based policy recommendations. The results update when the API data changes.

> **Presentation explanation:** We saved all tables and created one summary that can be copied into the report or slides.

In [ ]:
descriptive_statistics.to_csv(OUTPUT_DIR / 'world_bank_group1_descriptive_statistics.csv')
gdp_change.to_csv(OUTPUT_DIR / 'world_bank_group1_gdp_endpoint_changes.csv', index=False)
life_change.to_csv(OUTPUT_DIR / 'world_bank_group1_life_expectancy_endpoint_changes.csv', index=False)
education_change.to_csv(OUTPUT_DIR / 'world_bank_group1_education_endpoint_changes.csv', index=False)
rate_comparison.to_csv(OUTPUT_DIR / 'world_bank_group1_gdp_life_rate_comparison.csv', index=False)
balanced.to_csv(OUTPUT_DIR / 'world_bank_group1_balanced_improvement_scores.csv', index=False)
policy_recommendations.to_csv(OUTPUT_DIR / 'world_bank_group1_policy_recommendations.csv', index=False)

research_summary_rows = []
if not gdp_ranked.empty:
    leader = gdp_ranked.iloc[0]
    research_summary_rows.append({
        'research_question': 'Q1 - Highest GDP per capita growth',
        'evidence_statement': (
            f"{leader['country']} ranked first with {leader['percent_change']:.1f}% "
            f"growth between {int(leader['start_year'])} and {int(leader['end_year'])}. "
            f"Basis: {gdp_comparison_basis}."
        ),
    })
if not full_period_rate_comparison.empty:
    same_count = int(full_period_rate_comparison['same_rate_within_0_1_pp'].sum())
    research_summary_rows.append({
        'research_question': 'Q2 - GDP and life expectancy rates',
        'evidence_statement': (
            f'{same_count} of {len(full_period_rate_comparison)} countries in the completed 2010-2024 panel '
            'endpoints had annualised GDP and life-expectancy growth rates within 0.1 percentage '
            'points. One country required GDP imputation after 2015.'
        ),
    })
if not education_ranked.empty:
    leader = education_ranked.iloc[0]
    research_summary_rows.append({
        'research_question': 'Q3 - Greatest secondary-enrolment improvement',
        'evidence_statement': (
            f"{leader['country']} ranked first with a {leader['absolute_change']:.1f}-percentage-point "
            f"change between {int(leader['start_year'])} and {int(leader['end_year'])}. "
            f"Basis: {education_comparison_basis}. The original education series was 43.3% complete."
        ),
    })
if not correlation_results.empty:
    pearson_row = correlation_results.loc[correlation_results['method'].eq('Pearson_completed_panel')].iloc[0]
    reported_control = observed_regression_coefficients.loc[observed_regression_coefficients['term'].eq('log_gdp_per_capita')].iloc[0]
    significance = 'statistically significant' if pearson_row['p_value'] < 0.05 else 'not statistically significant'
    research_summary_rows.append({
        'research_question': 'Q4 - GDP and life-expectancy relationship',
        'evidence_statement': (
            f"The pooled Pearson correlation was r={pearson_row['coefficient']:.3f} "
            f"(p={pearson_row['p_value']:.3g}, n={int(pearson_row['n'])}), which was {significance} at alpha=0.05. "
            f'Reported-only controlled GDP p={reported_control["p_value"]:.3f}; results are model-sensitive.'
        ),
    })
if not balanced.empty:
    leader = balanced.iloc[0]
    research_summary_rows.append({
        'research_question': 'Q5 - Most balanced improvement',
        'evidence_statement': (
            f"{leader['country']} ranked first on the documented balance-adjusted index "
            f"with a score of {leader['balanced_improvement_score']:.1f}."
        ),
    })

research_summary_rows.append({
    'research_question': 'Q6 - Evidence-based policy recommendations',
    'evidence_statement': (
        f'{len(policy_recommendations)} recommendations were produced; each includes a '
        'named result and an explanation of why it supports the action.'
    ),
})

research_question_summary = pd.DataFrame(research_summary_rows)
research_question_summary.to_csv(
    OUTPUT_DIR / 'world_bank_group1_research_question_summary.csv', index=False
)
display(research_question_summary)


## Step 29 - Final verification

Before presentation, confirm that the clean CSV has zero missing analytical values, every estimate has a method flag, outlier candidates have a recorded review action, p-values are accompanied by effect sizes, and every policy recommendation cites a specific result.

> **Presentation explanation:** We checked that the completed dataset is fully populated, traceable, and supported by saved analysis outputs.


In [ ]:
required_figure_files = [
    '00b_data_treatment_summary.png',
    '01_gdp_per_capita_trends.png',
    '02_gdp_growth_ranking.png',
    '03_indexed_gdp_vs_life_expectancy.png',
    '04_secondary_enrolment_improvement.png',
    '05_gdp_life_expectancy_relationship.png',
    '06_balanced_improvement_heatmap.png',
]
final_checks = pd.DataFrame([
    {'check': 'Fully cleaned CSV exists', 'passed': (OUTPUT_DIR / 'world_bank_group1_cleaned_complete_2010_2024.csv').exists()},
    {'check': 'Cleaned analytical values contain no missing cells', 'passed': int(clean_df[ANALYSIS_COLUMNS].isna().sum().sum()) == 0},
    {'check': 'Every analytical value has a method label', 'passed': all(clean_df[f'{c}_method'].notna().all() for c in ANALYSIS_COLUMNS)},
    {'check': 'Imputation audit exists', 'passed': (OUTPUT_DIR / 'world_bank_group1_imputation_audit.csv').exists()},
    {'check': 'Outlier review exists', 'passed': (OUTPUT_DIR / 'world_bank_group1_outlier_review.csv').exists()},
    {'check': 'Three statistical result files exist', 'passed': all((OUTPUT_DIR / name).exists() for name in [
        'world_bank_group1_trend_analysis.csv',
        'world_bank_group1_correlation_results.csv',
        'world_bank_group1_regression_coefficients.csv',
        'world_bank_group1_regression_reported_only_coefficients.csv',
    ])},
    {'check': 'Required figures exist', 'passed': all((FIGURE_DIR / name).exists() for name in required_figure_files)},
    {'check': 'Research-question summary exists', 'passed': (OUTPUT_DIR / 'world_bank_group1_research_question_summary.csv').exists()},
    {'check': 'Policy recommendation table exists', 'passed': (OUTPUT_DIR / 'world_bank_group1_policy_recommendations.csv').exists()},
])
display(final_checks)
if not final_checks['passed'].all():
    print('Some verification checks failed. Read the earlier cell messages and rerun from Step 1.')
else:
    print('All cleaned-data, audit, statistical, visualisation, and recommendation outputs passed verification.')


## Step 30 - Optional: download all outputs as one ZIP file

The files remain in Google Drive even if this optional download cell is not run.

> **Presentation explanation:** We placed all outputs into one ZIP file so they can be downloaded together.

In [ ]:
from google.colab import files

zip_base = '/content/EAC_Group1_Complete_Analysis_2010_2024'
zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
files.download(zip_path)

## Final interpretation checklist

1. Use the completed 2010-2024 panel for like-for-like comparisons.
2. Never describe an imputed value as a reported World Bank observation; use the method and `was_imputed` flags.
3. Explain that internal gaps use within-country interpolation and edge gaps use a region-adjusted estimate.
4. Explain that an IQR flag is not automatically an error; correct only values that fail a plausibility check.
5. Report the correlation coefficient, p-value, sample size, controlled regression result, and the non-causal limitation together.
6. State that the balanced-improvement score is an analyst-designed, sensitivity-dependent index.
7. Support every recommendation with a named output table, statistic, or figure from this notebook.
8. Submit the notebook, raw JSON, fully cleaned CSV, imputation audit, dashboard, slides, and GitHub repository.
